# 1. Project Overview

This notebook is a toy, tabular reinforcement-learning project for a simplified dating/conversation setting. The goal is to learn a policy over a small discrete state space, not to produce real dating advice.

The RL agent does **not** generate text directly. The trained model is a Q-table: it maps a discrete conversation state to values for abstract actions/strategies such as `ask_question`, `be_playful`, or `suggest_date`.

The LLM is optional and is **not trained**. It can play two separate roles:

- **Text-to-state classifier:** convert a real message into a discrete RL state.
- **World-model/data generator:** generate possible state-action transitions that can be cached and reused.

Interface flow:

```text
real message
-> LLM classifier
-> discrete state
-> RL policy / Q-table
-> recommended abstract action
```

Training flow:

```text
state + action
-> environment/world model
-> next state + reward + done
-> Q-table update
```

Environment choices:

- **Handcoded simulator:** fast toy baseline environment for cheap RL experiments.
- **Live LLM world model:** expensive optional environment/data generator.
- **Cached LLM environment:** preferred LLM-based workflow. Generate transition data once, save it, then train RL from the cache without more API calls.

Training happens only when `learn=True` and the Q-table update line is executed. The Q-table is the trained model. The LLM is never fine-tuned in this notebook.


# 2. Imports and Configuration

Package installation is intentionally separate from imports. API keys are read from environment variables only. If Kimi credentials are missing, LLM-dependent cells print a clear message or raise a clear error instead of silently doing the wrong thing.


In [ ]:
# Optional install cell. Run only if these packages are missing.
# %pip install openai python-dotenv pandas


In [ ]:
import os
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

try:
    from openai import OpenAI
except Exception:
    OpenAI = None

random.seed(42)
np.random.seed(42)

KIMI_API_KEY = os.getenv("KIMI_API_KEY")
KIMI_BASE_URL = os.getenv("KIMI_BASE_URL", "https://api.moonshot.ai/v1")
KIMI_MODEL = os.getenv("KIMI_MODEL", "moonshot-v1-8k")

LLM_ENABLED = bool(KIMI_API_KEY and KIMI_BASE_URL and KIMI_MODEL and OpenAI is not None)
client = OpenAI(api_key=KIMI_API_KEY, base_url=KIMI_BASE_URL) if LLM_ENABLED else None

if LLM_ENABLED:
    print(f"LLM enabled with model: {KIMI_MODEL}")
else:
    print("LLM disabled. Set KIMI_API_KEY, KIMI_BASE_URL, and KIMI_MODEL to run LLM cells.")


# 3. State Space and Action Space

The state is deliberately simplified to keep the RL algorithm interpretable. This is not a real dating model; it is a toy environment for learning RL system design.

State dimensions:

- `interest` in `["low", "medium", "high"]`
- `stage` in `["opener", "chat", "date"]`
- `tone` in `["cold", "neutral", "warm"]`

There are `3 x 3 x 3 = 27` states and 8 abstract actions.


In [ ]:
INTERESTS = ["low", "medium", "high"]
STAGES = ["opener", "chat", "date"]
TONES = ["cold", "neutral", "warm"]

ACTIONS = [
    "ask_question",
    "give_compliment",
    "share_story",
    "be_playful",
    "be_direct",
    "suggest_date",
    "slow_down",
    "end_chat",
]

ACTION_GUIDANCE = {
    "ask_question": "Ask an open-ended question to keep the conversation moving.",
    "give_compliment": "Give a specific, light compliment without overdoing it.",
    "share_story": "Share a short personal story that builds rapport.",
    "be_playful": "Use playful banter or gentle teasing.",
    "be_direct": "Use direct but warm escalation.",
    "suggest_date": "Suggest a simple, low-pressure plan to meet.",
    "slow_down": "Give them space and respond calmly without pushing.",
    "end_chat": "End the chat politely instead of forcing the conversation.",
}

INTEREST_MAP = {name: i for i, name in enumerate(INTERESTS)}
STAGE_MAP = {name: i for i, name in enumerate(STAGES)}
TONE_MAP = {name: i for i, name in enumerate(TONES)}

N_INTEREST = len(INTERESTS)
N_STAGE = len(STAGES)
N_TONE = len(TONES)
N_STATES = N_INTEREST * N_STAGE * N_TONE
N_ACTIONS = len(ACTIONS)


def _coerce_index(value, labels):
    if isinstance(value, str):
        return labels.index(value)
    return int(value)


def state_to_id(interest, stage, tone):
    interest = _coerce_index(interest, INTERESTS)
    stage = _coerce_index(stage, STAGES)
    tone = _coerce_index(tone, TONES)
    return interest * (N_STAGE * N_TONE) + stage * N_TONE + tone


def id_to_state(state_id):
    state_id = int(state_id)
    interest = state_id // (N_STAGE * N_TONE)
    rest = state_id % (N_STAGE * N_TONE)
    stage = rest // N_TONE
    tone = rest % N_TONE
    return interest, stage, tone


def describe_state(state_id):
    interest, stage, tone = id_to_state(state_id)
    return f"interest={INTERESTS[interest]}, stage={STAGES[stage]}, tone={TONES[tone]}"


def clamp(value, low, high):
    return max(low, min(high, value))


print(f"States: {N_STATES}")
print(f"Actions: {N_ACTIONS}")


# 4. Reward Function

Reward is the learning signal. The Q-table is updated to prefer actions that lead to higher future reward.

This reward function is intentionally simple. Terminal outcomes dominate; otherwise, warmer/higher-progress states get a small shaping reward.


In [ ]:
def compute_reward(next_state_id, outcome=None):
    """Compute reward from the next state and optional terminal outcome."""
    if outcome == "date_success":
        return 30.0
    if outcome in {"date_fail", "date_failed", "ghosted", "ended", "ended_by_agent"}:
        return -5.0

    interest, stage, tone = id_to_state(next_state_id)
    state_quality = 0.15 * interest + 0.25 * stage + 0.05 * tone
    time_cost = -0.10
    return state_quality + time_cost


# 5. Policy Helpers

`epsilon` is the exploration rate. With probability `epsilon`, the agent tries a random action; otherwise, it chooses the best action currently known in the Q-table.


In [ ]:
def choose_action(Q, state_id, epsilon):
    """Epsilon-greedy action selection."""
    if np.random.random() < epsilon:
        return int(np.random.randint(N_ACTIONS))
    return int(np.argmax(Q[state_id]))


def greedy_action(Q, state_id):
    return int(np.argmax(Q[state_id]))


def random_policy(state_id):
    return int(np.random.randint(N_ACTIONS))


def rule_based_action(state_id):
    interest, stage, tone = id_to_state(state_id)
    if interest == 2 and stage == 2 and tone == 2:
        return ACTIONS.index("suggest_date")
    if tone == 0:
        return ACTIONS.index("ask_question")
    if interest == 0:
        return ACTIONS.index("slow_down")
    if stage == 0:
        return ACTIONS.index("ask_question")
    if stage == 1 and tone == 2:
        return ACTIONS.index("be_playful")
    if stage == 1:
        return ACTIONS.index("share_story")
    return ACTIONS.index("be_direct")


# 6. Environment 1: Fast Handcoded Simulator

This environment is fast, deterministic/stochastic by handcoded rules, and useful for baseline training. It is not realistic; it is just a toy simulator.


In [ ]:
def reset_episode():
    interest = np.random.choice([0, 1], p=[0.55, 0.45])
    stage = 0
    tone = np.random.choice([0, 1], p=[0.35, 0.65])
    return state_to_id(interest, stage, tone)


def simulator_step_hardcoded(state_id, action):
    """Fast toy environment returning next_state_id, reward, done, outcome."""
    interest, stage, tone = id_to_state(state_id)
    done = False
    outcome = None

    if action == ACTIONS.index("ask_question"):
        interest += np.random.choice([0, 1], p=[0.55, 0.45])
        tone += np.random.choice([0, 1], p=[0.50, 0.50])
    elif action == ACTIONS.index("give_compliment"):
        if tone >= 1:
            interest += 1
        else:
            tone -= 1
    elif action == ACTIONS.index("share_story"):
        if stage >= 1:
            interest += np.random.choice([0, 1], p=[0.45, 0.55])
            tone += 1
    elif action == ACTIONS.index("be_playful"):
        if tone == 2:
            interest += 1
        else:
            tone += np.random.choice([-1, 1], p=[0.45, 0.55])
    elif action == ACTIONS.index("be_direct"):
        if interest >= 1 and tone >= 1:
            stage += 1
        else:
            interest -= 1
            tone -= 1
    elif action == ACTIONS.index("suggest_date"):
        done = True
        if interest == 2 and stage == 2 and tone == 2:
            outcome = "date_success" if np.random.random() < 0.75 else "date_fail"
        elif interest == 2 and tone == 2:
            outcome = "date_success" if np.random.random() < 0.40 else "date_fail"
        else:
            outcome = "date_fail"
    elif action == ACTIONS.index("slow_down"):
        tone += 1
        if interest == 0 and np.random.random() < 0.30:
            interest += 1
    elif action == ACTIONS.index("end_chat"):
        done = True
        outcome = "ended"

    if not done and np.random.random() < 0.03:
        done = True
        outcome = "ghosted"

    interest = clamp(interest, 0, N_INTEREST - 1)
    stage = clamp(stage, 0, N_STAGE - 1)
    tone = clamp(tone, 0, N_TONE - 1)
    next_state_id = state_to_id(interest, stage, tone)
    reward = compute_reward(next_state_id, outcome=outcome)
    return next_state_id, reward, done, outcome


# 7. Generic Episode Runner

This is where Q-learning actually happens when `learn=True`. The trained model is the Q-table, and the update below is the actual training step.


In [ ]:
def run_episode(
    env_step,
    Q=None,
    policy_fn=None,
    epsilon=0.0,
    max_steps=25,
    reset_fn=reset_episode,
    learn=False,
    alpha=0.1,
    gamma=0.95,
    trace=False,
):
    """Run one episode with any environment function."""
    state_id = reset_fn()
    total_reward = 0.0
    trace_rows = []
    outcome = "max_steps"

    for step in range(max_steps):
        if policy_fn is not None:
            action = int(policy_fn(state_id))
        elif Q is None:
            action = random_policy(state_id)
        else:
            action = choose_action(Q, state_id, epsilon)

        next_state_id, reward, done, outcome = env_step(state_id, action)
        total_reward += reward

        if learn:
            # This is the actual training step. The Q-table is the trained model.
            target = reward + (0.0 if done else gamma * np.max(Q[next_state_id]))
            Q[state_id, action] += alpha * (target - Q[state_id, action])

        if trace:
            trace_rows.append({
                "step": step + 1,
                "state_id": state_id,
                "state": describe_state(state_id),
                "action_id": action,
                "action": ACTIONS[action],
                "next_state_id": next_state_id,
                "next_state": describe_state(next_state_id),
                "reward": reward,
                "done": done,
                "outcome": outcome,
            })

        state_id = next_state_id
        if done:
            return total_reward, step + 1, outcome, trace_rows

    return total_reward, max_steps, outcome, trace_rows


# 8. Q-learning Training Function

`alpha` is the learning rate, `gamma` is the discount factor, and `epsilon` is the exploration rate. Q-learning is off-policy: it updates toward the best next action.


In [ ]:
def train_q_learning(
    env_step,
    episodes=8000,
    reset_fn=reset_episode,
    alpha=0.1,
    gamma=0.95,
    epsilon_start=1.0,
    epsilon_end=0.05,
    epsilon_decay_fraction=0.75,
):
    Q = np.zeros((N_STATES, N_ACTIONS))
    rewards = []

    for episode in range(episodes):
        progress = episode / max(1, episodes * epsilon_decay_fraction)
        epsilon = max(epsilon_end, epsilon_start - progress * (epsilon_start - epsilon_end))
        total_reward, _, _, _ = run_episode(
            env_step=env_step,
            Q=Q,
            epsilon=epsilon,
            reset_fn=reset_fn,
            learn=True,
            alpha=alpha,
            gamma=gamma,
        )
        rewards.append(total_reward)

    return Q, rewards


Q_hardcoded, rewards_hardcoded = train_q_learning(
    env_step=simulator_step_hardcoded,
    episodes=8000,
)
print("Q-learning hardcoded training complete.")


# 9. SARSA Training Function

SARSA is another tabular RL update rule, not another model type. It is on-policy: it updates toward the next action the current policy actually chooses.


In [ ]:
def run_episode_sarsa(
    env_step,
    Q,
    epsilon=0.0,
    max_steps=25,
    reset_fn=reset_episode,
    learn=False,
    alpha=0.1,
    gamma=0.95,
):
    state_id = reset_fn()
    action = choose_action(Q, state_id, epsilon)
    total_reward = 0.0
    outcome = "max_steps"

    for step in range(max_steps):
        next_state_id, reward, done, outcome = env_step(state_id, action)
        total_reward += reward

        if done:
            if learn:
                Q[state_id, action] += alpha * (reward - Q[state_id, action])
            return total_reward, step + 1, outcome

        next_action = choose_action(Q, next_state_id, epsilon)
        if learn:
            target = reward + gamma * Q[next_state_id, next_action]
            Q[state_id, action] += alpha * (target - Q[state_id, action])

        state_id = next_state_id
        action = next_action

    return total_reward, max_steps, outcome


def train_sarsa(
    env_step,
    episodes=8000,
    reset_fn=reset_episode,
    alpha=0.1,
    gamma=0.95,
    epsilon_start=1.0,
    epsilon_end=0.05,
    epsilon_decay_fraction=0.75,
):
    Q = np.zeros((N_STATES, N_ACTIONS))
    rewards = []

    for episode in range(episodes):
        progress = episode / max(1, episodes * epsilon_decay_fraction)
        epsilon = max(epsilon_end, epsilon_start - progress * (epsilon_start - epsilon_end))
        total_reward, _, _ = run_episode_sarsa(
            env_step=env_step,
            Q=Q,
            epsilon=epsilon,
            reset_fn=reset_fn,
            learn=True,
            alpha=alpha,
            gamma=gamma,
        )
        rewards.append(total_reward)

    return Q, rewards


Q_sarsa_hardcoded, rewards_sarsa_hardcoded = train_sarsa(
    env_step=simulator_step_hardcoded,
    episodes=8000,
)
print("SARSA hardcoded training complete.")


# 10. Policy Evaluation

All policy comparisons should use the same evaluation environment when you want direct comparisons. Results across different environments are not directly comparable.


In [ ]:
def make_greedy_policy(Q):
    return lambda state_id: greedy_action(Q, state_id)


def evaluate_policy(env_step, policy_fn, episodes=1000, reset_fn=reset_episode):
    rewards = []
    steps = []
    outcomes = []

    for _ in range(episodes):
        total_reward, length, outcome, _ = run_episode(
            env_step=env_step,
            policy_fn=policy_fn,
            reset_fn=reset_fn,
            learn=False,
        )
        rewards.append(total_reward)
        steps.append(length)
        outcomes.append(outcome)

    success_rate = sum(outcome == "date_success" for outcome in outcomes) / episodes
    return {
        "average_reward": float(np.mean(rewards)),
        "success_rate": success_rate,
        "avg_steps": float(np.mean(steps)),
        "outcome_counts": dict(pd.Series(outcomes).value_counts()),
    }


def compare_policies(rows):
    return pd.DataFrame(rows)


random_eval = evaluate_policy(simulator_step_hardcoded, random_policy)
rule_eval = evaluate_policy(simulator_step_hardcoded, rule_based_action)
q_eval = evaluate_policy(simulator_step_hardcoded, make_greedy_policy(Q_hardcoded))
sarsa_eval = evaluate_policy(simulator_step_hardcoded, make_greedy_policy(Q_sarsa_hardcoded))

baseline_results = compare_policies([
    {"policy_name": "Random", "training_env": "none", "evaluation_env": "handcoded", **random_eval},
    {"policy_name": "Rule-based", "training_env": "rules", "evaluation_env": "handcoded", **rule_eval},
    {"policy_name": "Q-learning", "training_env": "handcoded", "evaluation_env": "handcoded", **q_eval},
    {"policy_name": "SARSA", "training_env": "handcoded", "evaluation_env": "handcoded", **sarsa_eval},
])

baseline_results[["policy_name", "training_env", "evaluation_env", "average_reward", "success_rate", "avg_steps"]]


# 11. LLM Classifier Interface

This is the demo/deployment interface. A real message is converted into a discrete state, then the already-trained Q-table chooses an abstract strategy.

The classifier does **not** train the RL agent. It is only an interface layer.


In [ ]:
CLASSIFIER_SYSTEM_PROMPT = """You classify dating-app conversation messages into a toy RL state.
Return ONLY valid JSON with exactly: interest, stage, tone.

interest: low, medium, high
stage: opener, chat, date
tone: cold, neutral, warm

Be conservative. Short dry messages like "k" are usually low interest and cold tone. Without context, assume brief replies are part of chat unless clearly an opener.
"""


def _require_llm():
    if not LLM_ENABLED:
        raise RuntimeError("LLM is disabled. Set KIMI_API_KEY, KIMI_BASE_URL, and KIMI_MODEL first.")


def classify_message_with_llm(message, conversation_context=None):
    """Classify real text into interest/stage/tone labels."""
    _require_llm()
    context = conversation_context or "No extra context."
    user_prompt = f"Context:\n{context}\n\nMessage:\n{message}"
    response = client.chat.completions.create(
        model=KIMI_MODEL,
        messages=[
            {"role": "system", "content": CLASSIFIER_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0,
        response_format={"type": "json_object"},
    )
    data = json.loads(response.choices[0].message.content)
    expected = {"interest", "stage", "tone"}
    if set(data) != expected:
        raise ValueError(f"Expected {expected}, got {data}")
    if data["interest"] not in INTEREST_MAP:
        raise ValueError(f"Invalid interest: {data}")
    if data["stage"] not in STAGE_MAP:
        raise ValueError(f"Invalid stage: {data}")
    if data["tone"] not in TONE_MAP:
        raise ValueError(f"Invalid tone: {data}")
    return data


def recommend_action_from_message(message, Q=Q_hardcoded, conversation_context=None):
    state = classify_message_with_llm(message, conversation_context=conversation_context)
    state_id = state_to_id(state["interest"], state["stage"], state["tone"])
    action_id = greedy_action(Q, state_id)
    return {
        "message": message,
        "classified_state": state,
        "state_id": state_id,
        "recommended_action": ACTIONS[action_id],
        "suggested_next_move": ACTION_GUIDANCE[ACTIONS[action_id]],
    }


# 12. Environment 2: LLM World Model / Transition Generator

The LLM world model is not the trained agent. It is only being used to generate possible environment transitions.

**Do not rerun expensive LLM cells accidentally.** Each transition generation call uses the API. Start with tiny subsets before generating a full cache.


In [ ]:
LLM_TRANSITION_OUTCOMES = {None, "date_success", "date_fail", "ghosted", "ended"}


def _validate_transition_labels(data):
    if data["next_interest"] not in INTEREST_MAP:
        raise ValueError(f"Invalid next_interest: {data}")
    if data["next_stage"] not in STAGE_MAP:
        raise ValueError(f"Invalid next_stage: {data}")
    if data["next_tone"] not in TONE_MAP:
        raise ValueError(f"Invalid next_tone: {data}")
    if data["outcome"] not in LLM_TRANSITION_OUTCOMES:
        raise ValueError(f"Invalid outcome: {data}")


def generate_llm_transition(state_id, action, history=None, temperature=0.3):
    """Generate one state-action transition with Kimi and return a validated record."""
    _require_llm()
    interest, stage, tone = id_to_state(state_id)
    action_name = ACTIONS[action]
    history_text = json.dumps(history or [], ensure_ascii=False)

    if action_name == "end_chat":
        next_state_id = state_id
        outcome = "ended"
        return {
            "state_id": state_id,
            "state": describe_state(state_id),
            "action_id": action,
            "action": action_name,
            "next_state_id": next_state_id,
            "next_state": describe_state(next_state_id),
            "reward": compute_reward(next_state_id, outcome=outcome),
            "done": True,
            "outcome": outcome,
            "response_text": "",
            "reason": "Agent ended the chat.",
        }

    system_prompt = """You are a realistic dating-app world model for a toy RL environment.
Given a discrete state and an abstract agent action, simulate the other person's next reply and next discrete state.
Be skeptical about premature date suggestions. Use only the allowed labels.
Return ONLY JSON."""

    user_prompt = f"""Current state:
interest={INTERESTS[interest]}, stage={STAGES[stage]}, tone={TONES[tone]}

Agent action:
{action_name}: {ACTION_GUIDANCE[action_name]}

Conversation history:
{history_text}

Return JSON with exactly:
{{
  "response_text": "...",
  "next_interest": "low|medium|high",
  "next_stage": "opener|chat|date",
  "next_tone": "cold|neutral|warm",
  "done": false,
  "outcome": null | "date_success" | "date_fail" | "ghosted" | "ended",
  "reason": "short explanation"
}}"""

    last_error = None
    for attempt in range(2):
        try:
            response = client.chat.completions.create(
                model=KIMI_MODEL,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=temperature,
                response_format={"type": "json_object"},
            )
            data = json.loads(response.choices[0].message.content)
            expected = {"response_text", "next_interest", "next_stage", "next_tone", "done", "outcome", "reason"}
            if set(data) != expected:
                raise ValueError(f"Bad transition keys: {data}")
            _validate_transition_labels(data)
            break
        except Exception as exc:
            last_error = exc
            if attempt == 0:
                print(f"Transition generation failed once, retrying: {exc}")
                time.sleep(1.0)
                continue
            raise RuntimeError(f"LLM transition failed twice: {last_error}") from exc

    next_state_id = state_to_id(data["next_interest"], data["next_stage"], data["next_tone"])
    outcome = data["outcome"]
    reward = compute_reward(next_state_id, outcome=outcome)
    return {
        "state_id": state_id,
        "state": describe_state(state_id),
        "action_id": action,
        "action": action_name,
        "next_state_id": next_state_id,
        "next_state": describe_state(next_state_id),
        "reward": reward,
        "done": bool(data["done"]) or outcome is not None,
        "outcome": outcome,
        "response_text": data["response_text"],
        "reason": data["reason"],
    }


# 13. Generate and Save Cached LLM Transitions

A full cache can require many API calls: `27 states x 8 actions x 1 sample = 216 calls`. Use a small subset first.

**Do not rerun this section accidentally.** The cache is saved after every successful call so work is not lost.


In [ ]:
CACHE_PATH = Path("llm_transition_cache.jsonl")


def save_jsonl(records, path):
    path = Path(path)
    with path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    print(f"Saved {len(records)} records to {path}")


def load_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def generate_transition_cache(
    samples_per_state_action=1,
    state_ids=None,
    action_ids=None,
    cache_path=CACHE_PATH,
    sleep_between_calls=1.0,
    resume=True,
):
    _require_llm()
    cache_path = Path(cache_path)
    records = load_jsonl(cache_path) if resume else []
    existing_keys = {record["key"] for record in records if "key" in record}

    state_ids = list(range(N_STATES)) if state_ids is None else list(state_ids)
    action_ids = list(range(N_ACTIONS)) if action_ids is None else list(action_ids)
    calls_made = 0

    for state_id in state_ids:
        for action_id in action_ids:
            for sample_idx in range(samples_per_state_action):
                key = f"{state_id}:{action_id}:{sample_idx}"
                if key in existing_keys:
                    continue
                record = generate_llm_transition(state_id, action_id)
                record["key"] = key
                record["sample_idx"] = sample_idx
                records.append(record)
                existing_keys.add(key)
                with cache_path.open("a", encoding="utf-8") as f:
                    f.write(json.dumps(record, ensure_ascii=False) + "\n")
                calls_made += 1
                if calls_made % 10 == 0:
                    print(f"Generated {calls_made} new transitions; total cached={len(records)}")
                time.sleep(sleep_between_calls)

    print(f"Cache now has {len(records)} records at {cache_path}")
    return records


In [ ]:
# Safe starter example. This spends API calls only if LLM_ENABLED is True.
# Comment this out if you do not want to spend prompts right now.
if LLM_ENABLED:
    demo_records = generate_transition_cache(
        samples_per_state_action=1,
        state_ids=[state_to_id("medium", "chat", "neutral")],
        action_ids=[ACTIONS.index("ask_question"), ACTIONS.index("be_playful")],
        sleep_between_calls=1.0,
    )
else:
    print("LLM disabled; skipping demo transition cache generation.")


# 14. Build Cached LLM Environment

This cached environment lets the RL agent train for thousands of episodes without calling the LLM again.


In [ ]:
def build_transition_index(records):
    transition_index = {}
    for record in records:
        key = (int(record["state_id"]), int(record["action_id"]))
        transition_index.setdefault(key, []).append(record)
    return transition_index


def make_cached_llm_env_step(transition_index, fallback_env_step=simulator_step_hardcoded):
    def env_step(state_id, action):
        candidates = transition_index.get((state_id, action), [])
        if candidates:
            record = random.choice(candidates)
            return (
                int(record["next_state_id"]),
                float(record["reward"]),
                bool(record["done"]),
                record["outcome"],
            )
        return fallback_env_step(state_id, action)
    return env_step


# 15. Train Q-learning on Cached LLM Transitions

This trains from cached LLM-generated transitions without making new LLM calls. If the cache is tiny, this is only a workflow demo, not meaningful training.


In [ ]:
records = load_jsonl(CACHE_PATH)
transition_index = build_transition_index(records)
simulator_step_cached_llm = make_cached_llm_env_step(transition_index)

Q_cached_llm = None
rewards_cached_llm = []

if len(records) < 50:
    print("Cache is very small. This is only a demo. Generate more transition records for meaningful training.")
else:
    Q_cached_llm, rewards_cached_llm = train_q_learning(
        env_step=simulator_step_cached_llm,
        episodes=8000,
    )
    print("Cached LLM-environment Q-learning complete.")


# 16. Compare Policies Across Environments

These are different environments, so results are not directly comparable unless evaluated on the same environment. The table includes training and evaluation environment labels to make that distinction explicit.


In [ ]:
comparison_rows = []

comparison_specs = [
    ("Random", "none", random_policy, "handcoded", simulator_step_hardcoded),
    ("Rule-based", "rules", rule_based_action, "handcoded", simulator_step_hardcoded),
    ("Q_hardcoded", "handcoded", make_greedy_policy(Q_hardcoded), "handcoded", simulator_step_hardcoded),
    ("SARSA_hardcoded", "handcoded", make_greedy_policy(Q_sarsa_hardcoded), "handcoded", simulator_step_hardcoded),
]

if Q_cached_llm is not None:
    comparison_specs.extend([
        ("Q_cached_llm", "cached_llm", make_greedy_policy(Q_cached_llm), "cached_llm", simulator_step_cached_llm),
        ("Q_cached_llm", "cached_llm", make_greedy_policy(Q_cached_llm), "handcoded", simulator_step_hardcoded),
        ("Q_hardcoded", "handcoded", make_greedy_policy(Q_hardcoded), "cached_llm", simulator_step_cached_llm),
    ])
else:
    print("Skipping cached-LLM policy comparisons because Q_cached_llm has not been trained yet.")

for policy_name, training_env, policy_fn, eval_env_name, env_step in comparison_specs:
    result = evaluate_policy(env_step, policy_fn, episodes=1000)
    comparison_rows.append({
        "policy_name": policy_name,
        "training_env": training_env,
        "evaluation_env": eval_env_name,
        "avg_reward": result["average_reward"],
        "success_rate": result["success_rate"],
        "avg_steps": result["avg_steps"],
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df


# 17. Optional: Live LLM World-Model Training

This is slow and expensive because each episode can make many LLM calls. Use only for debugging or very small experiments. Prefer cached transitions for actual training loops.


In [ ]:
def live_llm_env_step_with_empty_history(state_id, action):
    """Compatibility wrapper for tiny live-LLM debugging only."""
    next_state_id, reward, done, outcome, _ = generate_llm_transition_as_step(state_id, action)
    return next_state_id, reward, done, outcome


def generate_llm_transition_as_step(state_id, action):
    record = generate_llm_transition(state_id, action)
    return (
        record["next_state_id"],
        record["reward"],
        record["done"],
        record["outcome"],
        record["response_text"],
    )


# Tiny example only. Uncomment intentionally if you want to spend API calls.
# if LLM_ENABLED:
#     Q_live_debug, rewards_live_debug = train_q_learning(
#         env_step=lambda s, a: generate_llm_transition_as_step(s, a)[:4],
#         episodes=2,
#     )


# 18. Limitations

- The state space is a toy abstraction.
- The rewards are hand-designed.
- This is not real dating advice.
- The LLM world model may be biased, noisy, or prompt-sensitive.
- Cached transitions are only as good as the prompt and generated data.
- There is no real human outcome data here.
- The goal is to learn RL system design with a transparent tabular agent.
